# Aegis GRPO Training Notebook

This notebook reruns the Hugging Face TRL training path used in the repo and writes a fresh evidence bundle with loss and reward plots.

It is designed to work both in a local Jupyter session and in Colab after the repository has been mounted or cloned into the runtime.

## Runtime Notes

- This notebook is meant to be run on **GPU** (Colab T4/L4/A10). It defaults to `Qwen/Qwen3-0.6B`.
- Tool execution in GRPO depends on a compatible chat template; we force-clone a known-good tool-calling template during training.
- The notebook writes artifacts to `reports/training_evidence/` and checkpoints to `artifacts/grpo-evidence/`.
- If you must run on CPU, keep the smoke run only (the 100-step run will be very slow).

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path.cwd()
if not (REPO_DIR / 'pyproject.toml').exists():
    raise RuntimeError('Run this notebook from the aegis repo root or clone/mount the repo before continuing.')

os.chdir(REPO_DIR)
print(f'Repo root: {REPO_DIR}')

In [ ]:
%pip install -q -e ".[openenv,server,eval,demo,training]"

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'training.train', '--check-stack'], check=True)

In [ ]:
import shutil
import subprocess
import sys
import torch

has_gpu = torch.cuda.is_available()
model_name = "Qwen/Qwen3-0.6B" if has_gpu else "Qwen/Qwen3-0.6B"  # CPU is supported but will be slow.
output_dir = "artifacts/grpo-evidence"
evidence_dir = "reports/training_evidence"

shutil.rmtree(output_dir, ignore_errors=True)
shutil.rmtree(evidence_dir, ignore_errors=True)

# 1) Quick smoke run (verifies wiring end-to-end)
smoke_cmd = [
    sys.executable,
    "-m",
    "training.train",
    "--train",
    "--model-name",
    model_name,
    "--episodes-per-attack",
    "1",
    "--max-steps",
    "5",
    "--per-device-train-batch-size",
    "2",
    "--gradient-accumulation-steps",
    "1",
    "--num-generations",
    "2",
    "--max-completion-length",
    "96",
    "--max-tool-calling-iterations",
    "6",
    "--logging-steps",
    "1",
    "--save-steps",
    "10",
    "--output-dir",
    output_dir,
    "--evidence-dir",
    evidence_dir,
    "--run-name",
    "aegis-grpo-smoke",
    "--force-clone-tool-template",
    "--log-completions",
]

print("Running smoke training...")
subprocess.run(smoke_cmd, check=True)

# 2) Evidence run (optimized for non-degenerate signals within ~100 steps)
if has_gpu:
    evidence_cmd = [
        sys.executable,
        "-m",
        "training.train",
        "--train",
        "--model-name",
        model_name,
        "--episodes-per-attack",
        "1",
        "--output-dir",
        output_dir,
        "--evidence-dir",
        evidence_dir,
        "--run-name",
        "aegis-grpo-evidence",
        "--per-device-train-batch-size",
        "4",
        "--fast-evidence-100",
    ]
    print("Running 100-step evidence training (GPU)...")
    subprocess.run(evidence_cmd, check=True)
else:
    print("No GPU detected; skipping the 100-step evidence run.")

print(f"Finished. Checkpoints: {output_dir} | Evidence: {evidence_dir}")

In [ ]:
import json
from pathlib import Path

summary_path = Path('reports/training_evidence/training_summary.json')
history_path = Path('reports/training_evidence/training_log_history.json')

summary = json.loads(summary_path.read_text(encoding='utf-8'))
history = json.loads(history_path.read_text(encoding='utf-8'))
summary, history

In [ ]:
from IPython.display import Image, display

display(Image(filename='reports/training_evidence/training_curves.png'))